In [22]:
import eval7
import random
import numpy as np

RANKS = "23456789TJQKA"
SUITS = "shdc"

deck_cards = [r + s for r in RANKS for s in SUITS]

def monte_carlo_equity(hole, board, n_players=2, iters=500):
    hero = [eval7.Card(c) for c in hole]
    board_cards = [eval7.Card(c) for c in board]
    dead = set(hole + board)
    wins = 0.0
    for _ in range(iters):
        deck = [c for c in deck_cards if c not in dead]
        random.shuffle(deck)
        idx = 0
        villains = []
        for _ in range(n_players - 1):
            v = [eval7.Card(deck[idx]), eval7.Card(deck[idx + 1]),]
            villains.append(v)
            idx += 2
        remain = 5 - len(board)
        runout = [eval7.Card(deck[idx + i]) for i in range(remain)]
        full_board = board_cards + runout
        hero_score = eval7.evaluate(hero + full_board)
        villain_scores = [eval7.evaluate(v + full_board) for v in villains]
        best = max(villain_scores)
        if hero_score > best:
            wins += 1.0
        elif hero_score == best:
            wins += 0.5
    return wins / iters

def equity_histogram(hole, board, n_players=2, bins=10, samples=200):
    equities = []
    for _ in range(samples):
        eq = monte_carlo_equity(hole, board, n_players=n_players, iters=50)
        equities.append(eq)
    hist, _ = np.histogram(equities, bins=bins, range=(0.0, 1.0), density=True)
    return hist

In [23]:
# Preflop bucketing
import itertools
import pickle
from tqdm import tqdm
from sklearn.cluster import KMeans

for k in range(2, 10):
    dataset = []
    all_cards = [r+s for r in "23456789TJQKA" for s in "shdc"]
    for hole in tqdm(itertools.combinations(all_cards, 2)):
        feat = equity_histogram(list(hole), [], n_players=k, bins=20, samples=100)
        dataset.append({"hole": hole, "features": feat,})
    with open(f"{k}_player_preflop_features.pkl", "wb") as f:
        pickle.dump(dataset, f)
        
    with open(f"{k}_player_preflop_features.pkl", "rb") as f:
        data = pickle.load(f)
    X = np.array([d["features"] for d in data])
    kmeans = KMeans(n_clusters=32, random_state=1,)
    labels = kmeans.fit_predict(X)
    lookup = {}
    for d, label in zip(data, labels):
        lookup[d["hole"]] = int(label)

    with open(f"{k}_player_preflop_buckets.pkl", "wb") as f:
        pickle.dump(lookup, f)

1326it [01:35, 13.84it/s]
1326it [02:23,  9.26it/s]
1326it [02:36,  8.48it/s]
1326it [02:32,  8.69it/s]
1326it [02:56,  7.52it/s]
1326it [02:02, 10.85it/s]
1326it [02:26,  9.06it/s]
1326it [02:01, 10.90it/s]


In [24]:
import random
import math

def unrank_combination(cards, k, rank):
    """
    Convert lexicographic rank -> k-combination of cards
    """
    n = len(cards)
    result = []
    x = rank
    start = 0
    for i in range(k):
        for j in range(start, n - (k - i) + 1):
            c = math.comb(n - j - 1, k - i - 1)
            if x < c:
                result.append(cards[j])
                start = j + 1
                break
            x -= c
    return tuple(result)

def sample_unique_combinations(all_cards, k=5, m=40_000):
    n = len(all_cards)
    total = math.comb(n, k)
    if m > total:
        raise ValueError("Requested more samples than total combinations")
    # sample unique combination indices
    indices = random.sample(range(total), m)
    return [unrank_combination(all_cards, k, idx) for idx in indices]

In [25]:
# Flop bucketing
import itertools
import pickle
from tqdm import tqdm
from sklearn.cluster import KMeans

for k in range(2, 10):
    dataset = []
    all_cards = [r+s for r in "23456789TJQKA" for s in "shdc"]
    sample = sample_unique_combinations(all_cards, 5, 20_000)
    for cards in tqdm(sample):
        hole, board = cards[:2], cards[2:]
        feat = equity_histogram(list(hole), list(board), n_players=k, bins=20, samples=100)
        dataset.append({"hole": hole, "features": feat,})
    with open(f"{k}_player_flop_features.pkl", "wb") as f:
        pickle.dump(dataset, f)
        
    with open(f"{k}_player_flop_features.pkl", "rb") as f:
        data = pickle.load(f)
    X = np.array([d["features"] for d in data])
    kmeans = KMeans(n_clusters=400, random_state=1,)
    labels = kmeans.fit_predict(X)
    lookup = {}
    for d, label in zip(data, labels):
        lookup[d["hole"]] = int(label)

    with open(f"{k}_player_flop_buckets.pkl", "wb") as f:
        pickle.dump(lookup, f)

100%|██████████| 20000/20000 [27:44<00:00, 12.02it/s]


In [26]:
# Turn bucketing
import itertools
import pickle
from tqdm import tqdm
from sklearn.cluster import KMeans

for k in range(2, 10):
    dataset = []
    all_cards = [r+s for r in "23456789TJQKA" for s in "shdc"]
    sample = sample_unique_combinations(all_cards, 6, 15_000)
    for cards in tqdm(sample):
        hole, board = cards[:2], cards[2:]
        feat = equity_histogram(list(hole), list(board), n_players=k, bins=20, samples=100)
        dataset.append({"hole": hole, "features": feat,})
    with open(f"{k}_player_turn_features.pkl", "wb") as f:
        pickle.dump(dataset, f)
        
    with open(f"{k}_player_turn_features.pkl", "rb") as f:
        data = pickle.load(f)
    X = np.array([d["features"] for d in data])
    kmeans = KMeans(n_clusters=300, random_state=1,)
    labels = kmeans.fit_predict(X)
    lookup = {}
    for d, label in zip(data, labels):
        lookup[d["hole"]] = int(label)

    with open(f"{k}_player_turn_buckets.pkl", "wb") as f:
        pickle.dump(lookup, f)

100%|██████████| 15000/15000 [20:14<00:00, 12.35it/s]


In [27]:
# River bucketing
import itertools
import pickle
from tqdm import tqdm
from sklearn.cluster import KMeans

for k in range(2, 10):
    dataset = []
    all_cards = [r+s for r in "23456789TJQKA" for s in "shdc"]
    sample = sample_unique_combinations(all_cards, 7, 10_000)
    for cards in tqdm(sample):
        hole, board = cards[:2], cards[2:]
        feat = equity_histogram(list(hole), list(board), n_players=k, bins=20, samples=100)
        dataset.append({"hole": hole, "features": feat,})
    with open(f"{k}_player_river_features.pkl", "wb") as f:
        pickle.dump(dataset, f)
        
    with open(f"{k}_player_river_features.pkl", "rb") as f:
        data = pickle.load(f)
    X = np.array([d["features"] for d in data])
    kmeans = KMeans(n_clusters=200, random_state=1,)
    labels = kmeans.fit_predict(X)
    lookup = {}
    for d, label in zip(data, labels):
        lookup[d["hole"]] = int(label)

    with open(f"{k}_player_river_buckets.pkl", "wb") as f:
        pickle.dump(lookup, f)

100%|██████████| 10000/10000 [13:05<00:00, 12.73it/s]
